**Linda Zier**

**ST 554**

**Final Project**

**Goal**

For this project we :

*   added our project and files to our github repo, committing often to show our progress.
*   wrote a Jupyter notebook that fits a machine learning model using pyspark’s MLlib module. In that same notebook we wrote code to read in a stream of data (data that we produced ourselves using a .py file that is also kept in the repo).
*   we used the model to do predictions on the stream and wrote those out to the console.


**Data**

The data is modified from the UCI machine learning repository. The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.


*   We used a chunk to build our model.
*   We then 'streamed data' to a folder that we monitored. As data came in we used our fitted model to predict on the incoming data.





# Fitting the Model

We created a Jupyter notebook for the model fitting part and the streaming part below. We completed the following:

*   read the data into a standard pandas data frame using the pd.read_csv() function
*   converted this to a spark data frame
*   treated the Power_Zone_3 variable as our response variable and used the other variables as predictors

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 07:40:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Creating the Pipeline

We fit an elastic net model using CV with the steps below.
The transformations below each used an MLlib function that we put into a pipeline.

*   We used an SQL transformer to cast the hour variable as a DoubleType.

*   We binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).

*   The month column was one-hot encoded.
*   We Ran a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. We did this by:
    - first by using a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator
    
    - then we had a PCA transformer for use in our pipeline.
    - we used two PCs in our transformation.


*   We renamed our response variable as label

*   We used VectorAssembler() to put our predictors into a features. The predictors are:

    – two fitted PCA features

    – binary Hour variable

    – Power_Zone_1

    – Power_Zone_2

    – Month indicator variables


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

transformations complete


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

pipeline complete


### Fitting an Elastic Net Model
Next we used the CrossValidator() function and the LinearRegression() function to fit an elastic net model. We did multiple combinations of reg and elastic net parameters.  We fit the model using 5-fold cross validation with root mean square error (RMSE) as the criteria: we're training 5 separate models (one per fold) and averaging their RMSE's together to get their RMSE for that combination. We then report the optimal values chosen for the tuning parameters and the CV error which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regParam:", cvModel.bestModel.getRegParam())
print("Optimal elasticNetParam:", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE = ", min(cvModel.avgMetrics))


26/04/28 07:40:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 07:40:41 WARN Instrumentation: [92f5a5b2] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 07:40:42 WARN Instrumentation: [92f5a5b2] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 07:40:43 WARN Instrumentation: [167796fc] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 07:40:44 WARN Instrumentation: [167796fc] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 07:40:44 WARN Instrumentation: [f3a8778c] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 07:40:44 WARN Instrumentation: [f3a8778c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regParam: 0.75
Optimal elasticNetParam: 0.05
CV RMSE =  2147.9915450646813


Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set
• Take the outputted transformations from the model (the predictions) and create a residual column
(label - prediction). The .withColumn() method is handy here. Print out a data frame with these
residuals, the label column, and the predictions

In [10]:


# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.0986857226044
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20881.02309251836|-640.0592325183607|
|20131.08434|18658.629216636986|1472.4551233630154|
|19668.43373|18203.149907447772|1465.2838225522282|
|18899.27711| 17589.15135449324|1310.1257555067605|
|18442.40964| 16995.80576246181| 1446.603877538193|
|18130.12048|16516.236673341555|1613.8838066584467|
|17945.06024|16091.851423736553|1853.2088162634464|
|17459.27711|15721.321148554427| 1737.955961445572|
|17025.54217|15269.716745347276| 1755.825424652725|
|16794.21687|14937.032724510696|1857.1841454893038|
|16638.07229|14651.267373647177| 1986.804916352823|
|16395.18072|14413.804856015198| 1981.375863984802|
|16117.59036|14081.623009613315|2035.9673503866852|
| 15822.6506|13623.677990947792| 2198.972609052209|
|15672.28916|13449.244410849537|2223.0447491504638|
|15597.10843|13301.29547123994

# Handling Streaming Data
We've downloaded a file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in our final_project/data directory.  This will be our source of random sampling.
### Reading a Stream
We’ve read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data where I will read in my .csv files. The schema is set to that of the original data since that is what our incoming data will look like.Then we set up the readStream assuming a header is present.

### Transform/Aggregation Step
• Now, we’ll do two separate things on the stream and join them together:
– With your stream, use your model transformer to obtain predictions from the incoming data. On
the resulting predictions also create a residual column as noted in the previous section (return
only the label, prediction and residual columns from this part)
– We can use our stream more than once! With another transformation on the (original) stream,
modify the response variable to be called label.
– Now join your above transform with this stream based on the label variable which should be
common to both!
– Note 1: This is a little silly, but I want you to join two transformations of the stream and I don’t
want things to get too crazy
– Note 2: Each data frame is created from the same stream of data! You don’t need two streams,
you can use the same stream and just do two separate transformations on it, combining it with a
.join() method from one of the SQL style data frames you are dealing with (as we discussed in
the notes)
Writing Step
• Now write your stream to the console using the append output mode.
• Start the query!

In [11]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream
streamDF = spark.readStream.scchema(stream_schema).option("header", True)\
           .csv("streaming_data")

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])


AttributeError: 'DataStreamReader' object has no attribute 'scchema'